##This notebook is a stepping process through the validation processes involved with truth detection. This includes things like library importing, EDA, model training, API configuration, etc.

# Exploratory Walkthrough of Statistical Truth Detection
This serves to decide the 

## Step 0: Imports

In [1]:
# Upgrade pip and setuptools
!pip install --upgrade pip setuptools wheel

# core and NLP libraries
!pip install numpy pandas matplotlib scikit-learn nltk textblob spacy requests torch

# get teh fever datasets and pyarrow bc error with version control
!pip install \
    transformers==4.33.3 \
    keras==2.11.0 \
    tensorflow==2.11.0 \
    datasets==2.13.1 \
    fsspec==2023.6.0 \
    pyarrow==15.0.2 \
    huggingface_hub==0.17.3 \
    git+https://github.com/UKPLab/sentence-transformers.git@v2.2.2

# get the rag libraries
!pip install llama-index sentence-transformers faiss-cpu

# get the spacy English model
# gives URL error: !python -m spacy download en_core_web_sm
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

  Cloning https://github.com/UKPLab/sentence-transformers.git (to revision v2.2.2) to /private/var/folders/kd/8x50n3n57kl2gy8fx5cxjt9r0000gp/T/pip-req-build-96o50umj
  Running command git clone --filter=blob:none --quiet https://github.com/UKPLab/sentence-transformers.git /private/var/folders/kd/8x50n3n57kl2gy8fx5cxjt9r0000gp/T/pip-req-build-96o50umj
  Running command git checkout -q f38e91e1e505fb338f03ad80a2f2b473569c7591
  Resolved https://github.com/UKPLab/sentence-transformers.git to commit f38e91e1e505fb338f03ad80a2f2b473569c7591
  Preparing metadata (setup.py) ... done
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)


In [2]:
# imports
import importlib
import re
import requests
import spacy
import json
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
import numpy as np
from typing import List, Set
# for NLP
import nltk
# for tokenizing, using pretrained model (BERT, literature)
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
# sentence detection/classification
from textblob import TextBlob
# for RAG: https://developers.llamaindex.ai/python/framework/#introduction
import llama_index 
# for FEVER
import datasets
import pyarrow
# from datasets import load_dataset
# models/classification
import torch

# get FEVER
# this dataset as starting point but likely use the full json: https://huggingface.co/datasets?modality=modality:text&sort=trending&search=fever
# full json: https://fever.ai/dataset/fever.html

# get NLTK data
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('brown')

# download the scapy english model
nlp = spacy.load("en_core_web_sm")

# autoupdate 
%load_ext autoreload
%autoreload 2

print("All installations complete!")

2025-10-22 13:53:04.966902: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/admin/Documents/graduate/penn/courses/dats598/backend/.venv/lib/python3.10/site-packages/transformers/utils/generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
[nltk_data] Downloading package punkt to /Users/admin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/admin/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordne

All installations complete!


## Step 1: Load FEVER

In [3]:
from datasets import load_dataset

fever_configs = ['v1.0', 'v2.0', 'wiki_pages']
fever = load_dataset("fever", fever_configs[0], cache_dir="./data/hf_cache")
fev_train = fever["train"].to_pandas()
fev_train.head()

Found cached dataset fever (/Users/admin/Documents/graduate/penn/courses/dats598/backend/dev/data/hf_cache/fever/v1.0/1.0.0/7f8936e0558704771b08c7ce9cc202071b29a0050603374507ba61d23c00a58e)


  0%|          | 0/6 [00:00<?, ?it/s]

,id,label,claim,evidence_annotation_id,evidence_id,evidence_wiki_url,evidence_sentence_id
0,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,92206,104971,Nikolaj_Coster-Waldau,7
1,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,92206,104971,Fox_Broadcasting_Company,-1
2,150448,SUPPORTS,Roman Atwood is a content creator.,174271,187498,Roman_Atwood,1
3,150448,SUPPORTS,Roman Atwood is a content creator.,174271,187499,Roman_Atwood,3
4,214861,SUPPORTS,"History of art includes architecture, dance, s...",255136,254645,History_of_art,2


## Step 2: Claim Detection from Text

In [14]:
import test_claim_detection, test_claim_extraction
# test_claim_detection.main(n_claims=1000)
test_claim_extraction.main(n_claims=1000)

Found cached dataset fever (/Users/admin/Documents/graduate/penn/courses/dats598/backend/dev/data/hf_cache/fever/v1.0/1.0.0/7f8936e0558704771b08c7ce9cc202071b29a0050603374507ba61d23c00a58e)


  0%|          | 0/6 [00:00<?, ?it/s]

Average semantic similarity:


## Step 3: ER